# Naive to Advanced RAG

This notebook walks through a small retrieval pipeline from basic to more advanced patterns:

- Basic LCEL-style RAG
- Multi-query retrieval
- HyDE
- Parent-child retrieval
- Re-ranking with a cross-encoder / bge-reranker style model

LangChain’s retrieval docs describe retrieval as a modular pipeline where loaders, splitters, embeddings, and vector stores can be swapped independently, and retrievers are Runnables that can also work beyond vector stores.

## Learning goals

By the end of this notebook, you should be able to:

1. Build a naive vector-search-based RAG baseline.
2. Expand a query into multiple variants.
3. Use a HyDE-style hypothetical answer to improve retrieval.
4. Retrieve small chunks while keeping parent documents intact.
5. Add a reranking pass on top of the initial retrieval results.

## 1) Install packages

```bash
pip install -U sentence-transformers langchain langchain-community langchain-text-splitters langchain-huggingface faiss-cpu langchain-classic
```

The LangChain cross-encoder reranker guide shows the same general pattern: a base retriever returns a larger candidate set, then a cross-encoder reranks it down to the best few passages.

In [ ]:
%pip install -qU sentence-transformers langchain langchain-community langchain-text-splitters langchain-huggingface faiss-cpu langchain-classic

## 2) Create a tiny document set

To keep this lab simple, we will use a few short documents in memory instead of loading a large corpus.

In [ ]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content="Our platform stores prompt history, token usage, and latency logs in SQLite. Retrieval is used to find support notes and product documentation quickly.",
        metadata={"source": "doc_a", "topic": "platform"},
    ),
    Document(
        page_content="Customer support articles explain refunds, shipping, and billing. Search quality improves when the query is expanded using related terms.",
        metadata={"source": "doc_b", "topic": "support"},
    ),
    Document(
        page_content="Internal engineering notes describe indexing, chunking, reranking, and parent-child retrieval for longer technical documents.",
        metadata={"source": "doc_c", "topic": "engineering"},
    ),
    Document(
        page_content="A knowledge base works best when the retriever returns the most relevant passages and the final answer only uses that context.",
        metadata={"source": "doc_d", "topic": "rag"},
    ),
]

documents

## 3) Split into chunks

LangChain’s retrieval docs describe text splitters as a way to break larger documents into smaller chunks that fit model context windows.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=120,
    chunk_overlap=20,
)

chunks = splitter.split_documents(documents)

print("Chunk count:", len(chunks))
for i, chunk in enumerate(chunks, 1):
    print(f"--- Chunk {i} ---")
    print(chunk.page_content)
    print(chunk.metadata)

## 4) Create local CPU embeddings

We will use a small sentence-transformers model on CPU so the notebook stays lightweight.

LangChain’s Hugging Face docs say local embedding models can be run directly through the LangChain integration, and the embeddings latency docs note that local CPU models avoid network latency but still take compute time.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model ready")

## 5) Build a simple vector store

We will use FAISS for a simple local vector index.

This gives us a naive retriever baseline before we try more advanced strategies.

In [ ]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(chunks, embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

print("Vector store ready")

## 6) Naive RAG baseline

The naive version of RAG is straightforward:

1. Embed the question.
2. Retrieve the closest chunks.
3. Feed those chunks into the answer step later.

The retrieval docs describe this as the core retrieval-to-RAG pattern.

In [ ]:
query = "How do we improve search quality for long documents?"
docs = retriever.invoke(query)

for i, doc in enumerate(docs, 1):
    print(f"--- Retrieved {i} ---")
    print(doc.page_content)
    print(doc.metadata)

## 7) Basic RAG context builder

This is the simplest LCEL-style idea in plain Python:

- retrieve context
- join the chunks
- use that context to answer later

In a real app, the next step would be a chat model prompt.

In [ ]:
def build_context(query: str, k: int = 3) -> str:
    docs = vector_store.similarity_search(query, k=k)
    return "".join(
        f"[source={doc.metadata.get('source')}]: {doc.page_content}"
        for doc in docs
    )

context = build_context(query)
print(context)

## 8) Multi-query retrieval

A single user query can miss relevant passages if the wording is different from the source text.

A basic multi-query approach expands the question into a few variants and merges the results.

In [ ]:
def expand_query(query: str) -> list[str]:
    return [
        query,
        query.replace("search quality", "retrieval quality"),
        query.replace("long documents", "large documents"),
    ]

expanded_queries = expand_query(query)
expanded_queries

In [ ]:
def multi_query_retrieve(query: str, k: int = 2):
    seen = set()
    merged = []
    for q in expand_query(query):
        for doc in vector_store.similarity_search(q, k=k):
            key = (doc.metadata.get("source"), doc.page_content)
            if key not in seen:
                seen.add(key)
                merged.append(doc)
    return merged

mq_docs = multi_query_retrieve(query)
for i, doc in enumerate(mq_docs, 1):
    print(f"--- Multi-query {i} ---")
    print(doc.page_content)
    print(doc.metadata)

## 9) HyDE

HyDE means hypothetical document embeddings.

The simple idea is to create a hypothetical answer first, then embed that text and retrieve with it.

For this notebook, we will keep it very lightweight and use a generated hypothetical answer string.

In [ ]:
def hyde_hypothesis(query: str) -> str:
    return (
        f"A helpful answer to the question '{query}' would likely discuss retrieval quality, "
        "chunking, semantic matching, and the need to compare multiple candidate passages."
    )

hyde_text = hyde_hypothesis(query)
print(hyde_text)

hyde_docs = vector_store.similarity_search(hyde_text, k=3)
for i, doc in enumerate(hyde_docs, 1):
    print(f"--- HyDE {i} ---")
    print(doc.page_content)
    print(doc.metadata)

## 10) Parent-child retrieval

This pattern keeps small chunks for matching, but returns the larger parent document so you do not lose context.

The basic workflow is:

1. Split parent docs into child chunks.
2. Retrieve the child chunks.
3. Map each child back to its parent.
4. Return the parent documents.

The LangChain docs describe retrievers as Runnables and show that retrieval can be separated from the underlying storage layer.

In [ ]:
parent_documents = [
    Document(
        page_content="Parent A: This document explains chunking, semantic search, and reranking. Keep the full context when the answer needs more than one paragraph.",
        metadata={"parent_id": "A"},
    ),
    Document(
        page_content="Parent B: This document explains support content, billing, shipping, and policy retrieval. The main idea is to preserve source context while searching smaller chunks.",
        metadata={"parent_id": "B"},
    ),
]

parent_splitter = RecursiveCharacterTextSplitter(chunk_size=80, chunk_overlap=10)
child_chunks = parent_splitter.split_documents(parent_documents)

child_to_parent = {p.metadata["parent_id"]: p for p in parent_documents}

child_store = FAISS.from_documents(child_chunks, embeddings)
child_retriever = child_store.as_retriever(search_kwargs={"k": 2})

child_hits = child_retriever.invoke("Why keep full context for answers?")
parent_hits = []
seen_parent_ids = set()

for child in child_hits:
    pid = child.metadata.get("parent_id")
    if pid and pid not in seen_parent_ids:
        seen_parent_ids.add(pid)
        parent_hits.append(child_to_parent[pid])

for i, doc in enumerate(parent_hits, 1):
    print(f"--- Parent {i} ---")
    print(doc.page_content)
    print(doc.metadata)

## 11) Re-ranking

Reranking usually means: retrieve a larger candidate set first, then sort those candidates again with a more precise model.

LangChain’s cross-encoder reranker guide explains that a cross-encoder scores each query-document pair directly, and that this top-k then rerank pattern is a high-impact quality improvement for RAG. The guide also lists BAAI/bge-reranker models as supported examples.

In [ ]:
candidate_docs = vector_store.similarity_search(query, k=5)

def simple_rerank(query: str, docs):
    scored = []
    q_terms = set(query.lower().split())
    for doc in docs:
        text = doc.page_content.lower()
        score = sum(1 for term in q_terms if term in text)
        scored.append((score, doc))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [doc for score, doc in scored]

reranked = simple_rerank(query, candidate_docs)
for i, doc in enumerate(reranked, 1):
    print(f"--- Reranked {i} ---")
    print(doc.page_content)
    print(doc.metadata)

## 12) Optional real cross-encoder setup

If you want a stronger reranking pass, the LangChain reranker docs show a `HuggingFaceCrossEncoder` with `CrossEncoderReranker` and `ContextualCompressionRetriever`. A common local choice is a small BGE reranker model such as `BAAI/bge-reranker-v2-m3`.

In [ ]:
# Optional advanced setup (keep commented until you want to run it)
# from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
# from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
# from langchain_community.cross_encoders import HuggingFaceCrossEncoder
#
# cross_encoder = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-v2-m3")
# reranker = CrossEncoderReranker(model=cross_encoder, top_n=3)
# compression_retriever = ContextualCompressionRetriever(
#     base_compressor=reranker,
#     base_retriever=vector_store.as_retriever(search_kwargs={"k": 10}),
# )
# docs = compression_retriever.invoke(query)
# docs

print("Optional reranker template ready.")

## 13) Compare the strategies

A simple way to remember the progression:

- Naive RAG: one query, one retrieval step
- Multi-query: one question, several query variants
- HyDE: retrieve from a hypothetical answer embedding
- Parent-child: match on small chunks, return larger context
- Rerank: refine the shortlist with a stronger scoring pass

In [ ]:
summary = {
    "naive": len(retriever.invoke(query)),
    "multi_query": len(mq_docs),
    "hyde": len(hyde_docs),
    "parent_child": len(parent_hits),
    "reranked_candidates": len(reranked),
}
summary

## 14) Key takeaways

- Start with a naive retriever baseline.
- Query expansion can recover missed passages.
- HyDE is useful when the user query is short or underspecified.
- Parent-child retrieval preserves context while keeping matching chunks small.
- Reranking is often the biggest quality boost after the initial retrieval step.

LangChain’s retrieval docs frame RAG as modular, and the cross-encoder guide shows the standard retrieve-then-rerank pattern.

## References

- Retrieval: https://docs.langchain.com/oss/python/langchain/retrieval
- Semantic search / knowledge base: https://docs.langchain.com/oss/python/langchain/knowledge-base
- Retriever integrations: https://docs.langchain.com/oss/python/integrations/retrievers/index
- Cross encoder reranker: https://docs.langchain.com/oss/python/integrations/document_transformers/cross_encoder_reranker